# DQN traffic-signal control on real Pune traffic counts
This notebook reproduces the project end to end on Kaggle or Google Colab: it downloads the Pune dataset, builds 5-minute demand, trains a DQN agent in SUMO and compares it with fixed-time and actuated signal control.

In [ ]:
import os, subprocess, sys
REPO = os.environ.get('REPO_URL', 'https://github.com/MridulNegi2005/pune-signal-rl.git')
if not os.path.exists('pune-signal-rl'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
os.chdir('pune-signal-rl')
!pip -q install eclipse-sumo==1.23.1 libsumo==1.23.1 traci==1.23.1 stable-baselines3 gymnasium

## 1. Download the dataset (Mendeley Data, CC BY 4.0)

In [ ]:
!python src/download_data.py

## 2. Turn per-frame running totals into 5-minute counts

In [ ]:
!python src/preprocess.py
import pandas as pd
pd.read_csv('data/processed/pune_5min.csv').head()

## 3. Train the DQN agent
SUMO runs on the CPU in parallel copies; the Q-network trains on the GPU when one is present.

In [ ]:
import torch; print('GPU:', torch.cuda.is_available())
!cd src && python train.py --steps 150000 --envs 4 --seed 1 --name dqn_seed1

## 4. Test on unseen days, heavier demand, an unseen junction and free left turns

In [ ]:
!cd src && python evaluate.py --models ../results/dqn_seed1.zip
!cd src && python plots.py

In [ ]:
from IPython.display import Image, display
for f in ['demand_profile.png', 'learning_curve.png', 'comparison.png', 'wait_by_class.png']:
    display(Image('results/' + f))